In [1]:
import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD THE DATASET
# ==========================================
print("--- 1. LOADING DATASET ---")
file_path = 'project_traffic_dataset-v4.csv'
df = pd.read_csv(file_path)
print(f"Original Shape: {df.shape} (Rows: {df.shape[0]}, Columns: {df.shape[1]})\n")

# ==========================================
# 2. REMOVE DUPLICATE ROWS
# ==========================================
print("--- 2. REMOVING DUPLICATES ---")
df = df.drop_duplicates()
print(f"New Shape after removing duplicates: {df.shape}\n")

# ==========================================
# 3. CLEAN TEXT & TIME COLUMNS
# ==========================================
print("--- 3. CLEANING TEXT & TIME ---")

# Convert all text columns to lowercase and strip spaces
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.lower().str.strip()

# Standardize time columns if present
if 'time_exact' in df.columns:
    df['time_exact'] = df['time_exact'].str.lower()

if 'time_slot' in df.columns:
    df['time_slot'] = df['time_slot'].str.replace(' ', '').str.lower()

print("Text and time columns cleaned.\n")

# ==========================================
# 4. HANDLE MISSING VALUES
# ==========================================
print("--- 4. HANDLING MISSING VALUES ---")
if df.isnull().sum().sum() > 0:
    # Fill numeric columns with median
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].fillna(df[col].median())

    # Fill categorical columns with mode
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].fillna(df[col].mode()[0])

    print("Missing values filled successfully!\n")
else:
    print("No missing values found!\n")

# ==========================================
# 5. DATA VALIDATION CHECKS
# ==========================================
print("--- 5. VALIDATION CHECKS ---")

# A. Ensure no negative values in certain columns
non_negative_cols = ['distance_km', 'current_vehicle_count',
                     'avg_speed_kmh', 'potholes_count', 'traffic_lights_count']

for col in non_negative_cols:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)

print("✅ Negative values fixed (if any).")

# B. Ensure percentages are between 0 and 100
if 'two_wheeler_pct' in df.columns:
    df['two_wheeler_pct'] = df['two_wheeler_pct'].clip(0, 100)
if 'four_wheeler_pct' in df.columns:
    df['four_wheeler_pct'] = df['four_wheeler_pct'].clip(0, 100)

print("✅ Percentages validated within [0,100].")

# C. Show target variable distribution
if 'congestion_level' in df.columns:
    print("\nTarget Variable Distribution:")
    print(df['congestion_level'].value_counts())

# ==========================================
# 6. SAVE CLEANED DATASET
# ==========================================
output_file = 'project_traffic_dataset_cleaned.csv'
df.to_csv(output_file, index=False)
print(f"\n✅ Cleaned dataset saved as '{output_file}'")


--- 1. LOADING DATASET ---
Original Shape: (2799, 29) (Rows: 2799, Columns: 29)

--- 2. REMOVING DUPLICATES ---
New Shape after removing duplicates: (2799, 29)

--- 3. CLEANING TEXT & TIME ---
Text and time columns cleaned.

--- 4. HANDLING MISSING VALUES ---
No missing values found!

--- 5. VALIDATION CHECKS ---
✅ Negative values fixed (if any).
✅ Percentages validated within [0,100].

Target Variable Distribution:
congestion_level
high      933
low       933
medium    933
Name: count, dtype: int64

✅ Cleaned dataset saved as 'project_traffic_dataset_cleaned.csv'


In [2]:
import pymysql
import pandas as pd

# ==========================================
# 1. LOAD THE CLEANED DATASET
# ==========================================
file_path = "project_traffic_dataset_cleaned.csv"
df = pd.read_csv(file_path)
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# ==========================================
# 2. CONNECT TO MYSQL DATABASE
# ==========================================
try:
    conn = pymysql.connect(
        host="localhost",      # Change if using remote server
        user="root",           # Your MySQL username
        password="hk@121",     # Your MySQL password
        database="traffic" # Your database name
    )
    cursor = conn.cursor()
    print("✅ Connected to MySQL successfully!")
except Exception as e:
    print(f"❌ Database connection failed: {e}")
    exit()

# ==========================================
# 3. UPLOAD DATA ROW BY ROW
# ==========================================
# Example: Assume table name is 'traffic_data'
# Columns in CSV must match table columns in MySQL

table_name = "traffic_data"

# Create INSERT query dynamically
cols = ",".join(df.columns)  # column names
placeholders = ",".join(["%s"] * len(df.columns))  # %s for each column
insert_query = f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})"

try:
    # Upload each row
    for row in df.itertuples(index=False, name=None):
        cursor.execute(insert_query, row)

    conn.commit()
    print(f"✅ Uploaded {df.shape[0]} rows into '{table_name}' table!")

except Exception as e:
    print(f"❌ Error while uploading data: {e}")
    conn.rollback()

# ==========================================
# 4. CLOSE CONNECTION
# ==========================================
cursor.close()
conn.close()
print("🔒 MySQL connection closed.")


Dataset loaded: 2799 rows, 29 columns
✅ Connected to MySQL successfully!
❌ Error while uploading data: (1146, "Table 'traffic.traffic_data' doesn't exist")
🔒 MySQL connection closed.


In [3]:
import pandas as pd

# Load dataset just to get column names
file_path = "project_traffic_dataset_cleaned.csv"
df = pd.read_csv(file_path)

# Generate SQL CREATE TABLE statement
table_name = "traffic_data"
sql = f"CREATE TABLE IF NOT EXISTS {table_name} (\n"
sql += "    id INT AUTO_INCREMENT PRIMARY KEY,\n"

for col in df.columns:
    # Simple type inference: numeric → FLOAT, else VARCHAR
    if pd.api.types.is_numeric_dtype(df[col]):
        sql += f"    {col} FLOAT,\n"
    else:
        sql += f"    {col} VARCHAR(255),\n"

sql = sql.rstrip(",\n") + "\n);"

print(sql)


CREATE TABLE IF NOT EXISTS traffic_data (
    id INT AUTO_INCREMENT PRIMARY KEY,
    observation_id VARCHAR(255),
    route_id VARCHAR(255),
    route_name VARCHAR(255),
    source_city VARCHAR(255),
    destination_city VARCHAR(255),
    connected_routes VARCHAR(255),
    distance_km FLOAT,
    intermediate_traffic_spots VARCHAR(255),
    traffic_lights_count FLOAT,
    speed_limit_kmh FLOAT,
    time_exact VARCHAR(255),
    time_slot VARCHAR(255),
    time_period_category VARCHAR(255),
    is_festival_holiday FLOAT,
    is_weekend FLOAT,
    event_type VARCHAR(255),
    current_vehicle_count FLOAT,
    avg_historical_vehicle_count FLOAT,
    two_wheeler_pct FLOAT,
    four_wheeler_pct FLOAT,
    avg_speed_kmh FLOAT,
    gap_between_vehicles_m FLOAT,
    potholes_count FLOAT,
    cracks_present FLOAT,
    construction_active FLOAT,
    accident_reported FLOAT,
    petrol_pumps_count FLOAT,
    primary_congestion_reason VARCHAR(255),
    congestion_level VARCHAR(255)
);


In [4]:
import pymysql

conn = pymysql.connect(host="localhost", user="root", password="hk@121", database="traffic")
cursor = conn.cursor()

cursor.execute(sql)  # sql generated above
print("✅ Table created successfully!")

cursor.close()
conn.close()



✅ Table created successfully!


In [5]:
import pymysql

conn = pymysql.connect(host="localhost", user="root", password="hk@121", database="traffic")
cursor = conn.cursor()

table_name = "traffic_data"
cols = ",".join(df.columns)
placeholders = ",".join(["%s"] * len(df.columns))
insert_query = f"INSERT INTO {table_name} ({cols}) VALUES ({placeholders})"

for row in df.itertuples(index=False, name=None):
    cursor.execute(insert_query, row)

conn.commit()
print(f"✅ Uploaded {df.shape[0]} rows into '{table_name}' table!")

cursor.close()
conn.close()


✅ Uploaded 2799 rows into 'traffic_data' table!


In [7]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. LOAD DATA
# ============================================================

file_path = "project_traffic_dataset-v4.csv"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print("\nTarget Distribution:")
print(df["congestion_level"].value_counts())


# ============================================================
# 2. BASIC CLEANING
# ============================================================

# Remove duplicate rows
df = df.drop_duplicates()

# Remove rows where target is missing
df = df.dropna(subset=["congestion_level"])

print("\nAfter Cleaning:", df.shape)


# ============================================================
# 3. DEFINE TARGET
# ============================================================

TARGET = "congestion_level"

y = df[TARGET]


# ============================================================
# 4. SELECT FEATURES
# ============================================================

# IMPORTANT:
# observation_id is only an ID, so don't use it for prediction.
#
# primary_congestion_reason is excluded because it can represent
# information that may only be known after/along with congestion.
# This helps avoid data leakage.

features = [
    "route_id",
    "route_name",
    "source_city",
    "destination_city",
    "connected_routes",
    "distance_km",
    "intermediate_traffic_spots",
    "traffic_lights_count",
    "speed_limit_kmh",
    "time_exact",
    "time_slot",
    "time_period_category",
    "is_festival_holiday",
    "is_weekend",
    "event_type",
    "current_vehicle_count",
    "avg_historical_vehicle_count",
    "two_wheeler_pct",
    "four_wheeler_pct",
    "avg_speed_kmh",
    "gap_between_vehicles_m",
    "potholes_count",
    "cracks_present",
    "construction_active",
    "accident_reported",
    "petrol_pumps_count"
]

X = df[features]


# ============================================================
# 5. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# First:
# 70% Training
# 30% Temporary

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Now divide the remaining 30% into:
# 15% Validation
# 15% Test

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nData Split:")
print("Training   :", len(X_train))
print("Validation :", len(X_val))
print("Testing    :", len(X_test))


# ============================================================
# 6. IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ============================================================

categorical_features = [
    "route_id",
    "route_name",
    "source_city",
    "destination_city",
    "connected_routes",
    "intermediate_traffic_spots",
    "time_exact",
    "time_slot",
    "time_period_category",
    "event_type"
]

numerical_features = [
    "distance_km",
    "traffic_lights_count",
    "speed_limit_kmh",
    "is_festival_holiday",
    "is_weekend",
    "current_vehicle_count",
    "avg_historical_vehicle_count",
    "two_wheeler_pct",
    "four_wheeler_pct",
    "avg_speed_kmh",
    "gap_between_vehicles_m",
    "potholes_count",
    "cracks_present",
    "construction_active",
    "accident_reported",
    "petrol_pumps_count"
]


# ============================================================
# 7. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)


# ============================================================
# 8. RANDOM FOREST MODEL
# ============================================================

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)


# ============================================================
# 9. CREATE ML PIPELINE
# ============================================================

pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", model)
    ]
)


# ============================================================
# 10. TRAIN MODEL
# ============================================================

print("\nTraining Random Forest...")

pipeline.fit(X_train, y_train)

print("Training Completed!")


# ============================================================
# 11. VALIDATION
# ============================================================

val_prediction = pipeline.predict(X_val)

val_accuracy = accuracy_score(
    y_val,
    val_prediction
)

print("\n==============================")
print("VALIDATION RESULTS")
print("==============================")

print("Validation Accuracy:",
      round(val_accuracy * 100, 2),
      "%")

print("\nValidation Classification Report:")
print(
    classification_report(
        y_val,
        val_prediction
    )
)


# ============================================================
# 12. FINAL TEST
# ============================================================

test_prediction = pipeline.predict(X_test)

test_accuracy = accuracy_score(
    y_test,
    test_prediction
)

print("\n==============================")
print("FINAL TEST RESULTS")
print("==============================")

print("Test Accuracy:",
      round(test_accuracy * 100, 2),
      "%")

print("\nTest Classification Report:")
print(
    classification_report(
        y_test,
        test_prediction
    )
)


# ============================================================
# 13. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    test_prediction,
    labels=["Low", "Medium", "High"]
)

print("\nConfusion Matrix:")
print(cm)


# ============================================================
# 14. SAVE MODEL
# ============================================================

model_file = "traffic_congestion_model.joblib"

joblib.dump(
    pipeline,
    model_file
)

print("\nModel Saved Successfully!")
print("File:", model_file)


# ============================================================
# 15. TEST ONE SAMPLE
# ============================================================

sample = X_test.iloc[[0]]

prediction = pipeline.predict(sample)

probability = pipeline.predict_proba(sample)

print("\n==============================")
print("SAMPLE PREDICTION")
print("==============================")

print("Predicted Congestion:",
      prediction[0])

print("\nPrediction Probabilities:")

for class_name, prob in zip(
    pipeline.classes_,
    probability[0]
):
    print(
        class_name,
        ":",
        round(prob * 100, 2),
        "%"
    )

Dataset Shape: (2799, 29)

Target Distribution:
congestion_level
High      933
Low       933
Medium    933
Name: count, dtype: int64

After Cleaning: (2799, 29)

Data Split:
Training   : 1959
Validation : 420
Testing    : 420

Training Random Forest...
Training Completed!

VALIDATION RESULTS
Validation Accuracy: 93.1 %

Validation Classification Report:
              precision    recall  f1-score   support

        High       0.98      0.91      0.95       140
         Low       0.97      0.92      0.95       140
      Medium       0.85      0.96      0.90       140

    accuracy                           0.93       420
   macro avg       0.94      0.93      0.93       420
weighted avg       0.94      0.93      0.93       420


FINAL TEST RESULTS
Test Accuracy: 86.67 %

Test Classification Report:
              precision    recall  f1-score   support

        High       0.93      0.88      0.90       140
         Low       0.94      0.84      0.89       140
      Medium       0.76     

NameError: name 'congestion_level' is not defined

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Dataset load karo
df = pd.read_csv('project_traffic_dataset_cleaned.csv')

# 2. Missing values ko handle karo
df.fillna(df.mean(numeric_only=True), inplace=True)

# 3. Target column alag karo (pehle nikal lo, phir encoding karo)
target_col = 'congestion_level'
y = df[target_col]              # Target save kar lo
X = df.drop(target_col, axis=1) # Features alag kar lo

# 4. Categorical features ko encode karo (sirf X par)
X = pd.get_dummies(X, drop_first=True)

# 5. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6. Random Forest train karo
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 7. Predictions aur accuracy
predictions = model.predict(X_test)
print("Model Accuracy:", accuracy_score(y_test, predictions))
print("\nClassification Report:\n", classification_report(y_test, predictions))


Model Accuracy: 0.9303571428571429

Classification Report:
               precision    recall  f1-score   support

        high       0.88      0.91      0.90       180
         low       0.99      1.00      1.00       195
      medium       0.91      0.88      0.89       185

    accuracy                           0.93       560
   macro avg       0.93      0.93      0.93       560
weighted avg       0.93      0.93      0.93       560



In [14]:
import pandas as pd
# Model ko kya chizein sabse zaroori lag rahi hain:
feature_imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 5 Important Features:\n", feature_imp.head(10))

Top 5 Important Features:
 primary_congestion_reason_normal_flow        0.134296
gap_between_vehicles_m                       0.102032
avg_speed_kmh                                0.079893
current_vehicle_count                        0.062109
potholes_count                               0.049699
avg_historical_vehicle_count                 0.049393
primary_congestion_reason_severe_potholes    0.045216
accident_reported                            0.028023
two_wheeler_pct                              0.024762
four_wheeler_pct                             0.024028
dtype: float64


In [15]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Dataset load karein
df = pd.read_csv('project_traffic_dataset-v4.csv')

# 2. Jin features ko drop karna hai unki list
columns_to_drop = [
    'observation_id',
    'route_id',
    'time_exact',
    'accident_reported',
    'construction_active',
    'primary_congestion_reason',
]

# Unwanted columns ko drop karna
df_model = df.drop(columns=columns_to_drop)

# 3. Features (X) aur Target (y) ko alag karna
X = df_model.drop(columns=['congestion_level'])
y = df_model['congestion_level']

# 4. Categorical columns (text data) ko numbers me encode karna
categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
  le = LabelEncoder()
  X[col] = le.fit_transform(X[col].astype(str))

# Target variable (congestion_level) ko bhi encode karna
target_le = LabelEncoder()
y_encoded = target_le.fit_transform(y)

# 5. Training aur Testing data me split karna (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# 6. Random Forest Model train karna
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 7. Model ko test karna aur predictions nikalna
y_pred = rf_model.predict(X_test)

# 8. Accuracy aur Report print karna
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:\n')
print(
    classification_report(
        y_test, y_pred, target_names=target_le.classes_.astype(str)
    )
)

Accuracy: 0.8767857142857143

Classification Report:

              precision    recall  f1-score   support

        High       0.91      0.91      0.91       180
         Low       0.93      0.88      0.90       195
      Medium       0.79      0.85      0.82       185

    accuracy                           0.88       560
   macro avg       0.88      0.88      0.88       560
weighted avg       0.88      0.88      0.88       560



In [ ]:
`